In [32]:
import openml
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
import torch
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset


In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [39]:
CLASS_LABELS = ["walking", "upstairs", "downstairs", "sitting", "standing", "laying"]

dataset = openml.datasets.get_dataset(1478)
X, y, *_ = dataset.get_data(target=dataset.default_target_attribute)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Needed as separate array for KFold split
y_train_labels = y_train.astype(int) - 1  # Converts "5" -> 4

X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(pd.get_dummies(y_train).to_numpy(), dtype=torch.float32, device=device)

X_train.shape, y_train.shape

(torch.Size([8239, 561]), torch.Size([8239, 6]))

In [19]:
class MLP(nn.Module):
    def __init__(self, n_hidden, n_width):
        super().__init__()

        self.input_layer = nn.Linear(X.shape[1], n_width)
        self.hidden_layers = nn.ModuleList([nn.Linear(n_width, n_width) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(n_width, len(CLASS_LABELS))
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.relu(layer(x))
        x = self.output_layer(x)
        return x


In [20]:
def train_one_epoch(model, dataloader, optimizer, loss_fn) -> float:
    model.train()
    train_loss = 0.0

    for data, target in dataloader:
        optimizer.zero_grad()

        y_pred = model(data)

        loss = loss_fn(y_pred, target)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_loss /= len(dataloader)
    return train_loss

In [21]:
def eval_model(model, dataloader, loss_fn) -> float:
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for data, target in dataloader:
            y_pred = model(data)
            val_loss += loss_fn(y_pred, target).item()

    val_loss /= len(dataloader)
    return val_loss

In [42]:
def train_model(fold: int, model, train_loader, val_loader, optimizer, loss_fn, epochs: int):
    best_val_loss = 999999999.9

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_loss = eval_model(model, val_loader, loss_fn)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(model.state_dict(), f"../checkpoints/fold_{fold}.pth")

In [40]:
def train_one_model():
    model = MLP(2, 3)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    skf = StratifiedKFold(n_splits=5, shuffle=True)

    for train_index, val_index in skf.split(X_train, y_train_labels):
        train_ds = TensorDataset(X_train[train_index], y_train[train_index])
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

        val_ds = TensorDataset(X_train[val_index], y_train[val_index])
        val_loader = DataLoader(val_ds, batch_size=32, shuffle=True)
        train_model(0, model, train_loader, val_loader, optimizer, loss_fn, epochs=1)

    return model

In [43]:
train_one_model()

MLP(
  (input_layer): Linear(in_features=561, out_features=3, bias=True)
  (hidden_layers): ModuleList(
    (0-1): 2 x Linear(in_features=3, out_features=3, bias=True)
  )
  (output_layer): Linear(in_features=3, out_features=6, bias=True)
  (relu): ReLU()
)